In [2]:
import numpy as np
from datetime import datetime
from pyspark.sql import functions as F
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. CARGA DEL MODELO Y CONEXIÓN A TABLAS GOLD
# ---------------------------------------------------------
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Cargando modelo de embeddings '{MODEL_NAME}' para inferencia de preguntas...")
query_model = SentenceTransformer(MODEL_NAME)


# Cargar catálogo de chunks y sus embeddings correspondientes
print("Cargando vectores y metadatos de la Capa Gold...")
df_chunks = spark.table("gold_document_chunks")
df_embeddings = spark.table("gold_embeddings")

# Join para tener en memoria los textos, metadatos y vectores integrados
df_gold_joined = df_chunks.join(df_embeddings, on=["chunk_id", "document_id"], how="inner") \
    .select(
        "chunk_id",
        "document_id",
        "document_title",
        "document_category",
        "source_url",
        "page_number",
        "chunk_text",
        "embedding"
    )

# Recopilar los vectores a NumPy array para calcular Similitud Coseno vectorizada de alta velocidad
gold_data_cache = df_gold_joined.collect()

if gold_data_cache:
    # Matriz de vectores de la base de conocimientos (N x 384)
    embeddings_matrix = np.array([r.embedding for r in gold_data_cache], dtype=np.float32)
    # Normalización para asegurar producto escalar = similitud coseno exacta
    norms = np.linalg.norm(embeddings_matrix, axis=1, keepdims=True)
    norms[norms == 0] = 1e-10
    normalized_embeddings_matrix = embeddings_matrix / norms
    print(f" Matriz vectorial cargada e indexada en memoria: {len(gold_data_cache)} chunks disponibles.")
else:
    print(" Advertencia: La tabla gold_embeddings está vacía. Ejecuta la Fase 7 previamente.")

# ---------------------------------------------------------
# 2. FUNCIÓN REUTILIZABLE DE BÚSQUEDA SEMÁNTICA
# ---------------------------------------------------------

def search_documents(question: str, top_k: int = 5, category_filter: str = None):
    """
    Recibe una pregunta en texto plano, calcula su embedding y devuelve los top_k
    chunks más relevantes ordenados por score de similitud coseno (0.0 a 1.0).
    
    Parámetros:
    - question (str): Pregunta planteada por el usuario.
    - top_k (int): Número de resultados a devolver (por defecto 5).
    - category_filter (str): Opcional. Permite filtrar resultados por categoría ('hr', 'legal', etc.).
    """
    if not question or not question.strip():
        print("La pregunta no puede estar vacía.")
        return []
        
    if not gold_data_cache:
        print("No hay índice vectorial cargado.")
        return []

    # Tarea 2: Generar embedding de la pregunta
    query_vector = query_model.encode(question, normalize_embeddings=True)
    query_vector = np.array(query_vector, dtype=np.float32)

    # Tarea 3: Calcular similitud coseno (Producto punto de vectores normalizados)
    cosine_similarities = np.dot(normalized_embeddings_matrix, query_vector)

    # Construir lista de resultados con sus metadatos
    results = []
    for idx, similarity_score in enumerate(cosine_similarities):
        record = gold_data_cache[idx]
        
        # Filtro opcional por categoría
        if category_filter and record.document_category.lower() != category_filter.lower():
            continue
            
        results.append({
            "chunk_id": record.chunk_id,
            "document_id": record.document_id,
            "score": float(similarity_score),
            "document_title": record.document_title,
            "document_category": record.document_category,
            "source_url": record.source_url,
            "page_number": record.page_number,
            "chunk_text": record.chunk_text
        })

    # Tarea 4: Ordenar descendente por score y recortar a top_k
    results_sorted = sorted(results, key=lambda x: x["score"], reverse=True)[:top_k]

    # Tarea 5: Mostrar salida formateada
    print(f"\n Pregunta: {question}")
    print(f" Top {len(results_sorted)} resultados más relevantes:")
    print("-" * 80)
    
    for i, res in enumerate(results_sorted, 1):
        print(f"{i}. {res['document_title']} (Categoría: {res['document_category']}) - score: {res['score']:.2f}")
        print(f"   Fuente: {res['source_url']} (Pág. {res['page_number']})")
        # Mostrar extracto preview del texto del chunk (primeros 150 caracteres)
        preview_text = res['chunk_text'].replace('\n', ' ')[:150] + "..."
        print(f"   Texto: \"{preview_text}\"")
        print("-" * 80)

    return results_sorted

StatementMeta(, 9fc13d82-5835-4bdb-9539-c235e4f27275, 5, Finished, Available, Finished, False)

Cargando modelo de embeddings 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2' para inferencia de preguntas...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cargando vectores y metadatos de la Capa Gold...
 Matriz vectorial cargada e indexada en memoria: 489 chunks disponibles.


StatementMeta(, 9fc13d82-5835-4bdb-9539-c235e4f27275, 7, Finished, Available, Finished, False)

In [5]:
results_filtered = search_documents("¿Cuál es el procedimiento de compras?", top_k=3, category_filter="operations")

StatementMeta(, 9fc13d82-5835-4bdb-9539-c235e4f27275, 9, Finished, Available, Finished, False)


 Pregunta: ¿Cuál es el procedimiento de compras?
 Top 2 resultados más relevantes:
--------------------------------------------------------------------------------
1. Operations Procedimiento Compras Proveedores (Categoría: operations) - score: 0.30
   Fuente: abfss://Ecodocs@onelake.dfs.fabric.microsoft.com/LH_Ecodocs.Lakehouse/Files/Bronze/Documents/operations_procedimiento_compras_proveedores.txt (Pág. 1)
   Texto: "EcoPower Solutions S.L. - Procedimiento de Contratación y Compras.  Cualquier contratación de proveedores tecnológicos que superen los 15.000 euros an..."
--------------------------------------------------------------------------------
2. Operations Mantenimiento Correctivo (Categoría: operations) - score: 0.19
   Fuente: abfss://Ecodocs@onelake.dfs.fabric.microsoft.com/LH_Ecodocs.Lakehouse/Files/Bronze/Documents/operations_mantenimiento_correctivo.txt (Pág. 1)
   Texto: "EcoPower Solutions S.L. - Protocolo de Mantenimiento Correctivo de Equipos. El tiempo máximo de res